In [1]:
import polars as pl 
import seaborn as sns 
import matplotlib.pyplot as plt 
from dbconfig import engine 
print('Environment ready!')

Environment ready!


In [2]:
from pathlib import Path 

plots_dir = Path('plots')
plots_dir.mkdir(exist_ok = True)

In [3]:
pl.Config.set_tbl_rows(50)
pl.Config.set_tbl_cols(30)
pl.Config.set_fmt_str_lengths(80)
pl.Config.set_tbl_width_chars(180)
pl.Config.set_float_precision(2)
pl.Config.set_thousands_separator(",")

polars.config.Config

In [5]:
pl.read_database(
        query = """
        select table_name,
        column_name
        from information_schema.columns
        where table_schema = 'clean'
        order by table_name, ordinal_position;
        """, connection = engine
        )

table_name,column_name
str,str
"""categories""","""product_category_name"""
"""categories""","""product_category_name_english"""
"""customers""","""customer_id"""
"""customers""","""customer_unique_id"""
"""customers""","""customer_zip_code_prefix"""
"""customers""","""customer_city"""
"""customers""","""customer_state"""
"""order_items""","""order_id"""
"""order_items""","""order_item_id"""


In [12]:
seller_check = pl.read_database(
        query = """
        select seller_id,
        count(distinct order_id) as orders 
        from clean.order_items
        group by 1
        having count(distinct order_id) > 30;
        """, connection = engine 
        )

In [15]:
seller_check.describe()

statistic,seller_id,orders
str,str,f64
"""count""","""619""",619.00
"""null_count""","""0""",0.00
"""mean""",null,133.87
"""std""",null,205.50
"""min""","""001cca7ae9ae17fb1caed9dfb1094831""",31.00
"""25%""",null,45.00
"""50%""",null,70.00
"""75%""",null,131.00
"""max""","""fffd5413c0700ac820c7069d66d98c89""","1,854.00"


In [14]:
seller_check.quantile([0.8,0.9, 0.95])

seller_id,orders
str,list[f64]
null,"[158.00, 261.00, 402.00]"


In [29]:
seller_performance = pl.read_database(
        query = """
        select oi.seller_id,

        o.order_id,

        extract(
            day from (
                o.order_delivered_customer_date 
                - o.order_purchase_timestamp
                )
            ) as delivery_days,

        extract(
            day from (
                o.order_delivered_customer_date 
                - o.order_estimated_delivery_date
                )
            ) as delay_days,

        extract(
            day from (
                o.order_delivered_carrier_date
                - o.order_purchase_timestamp
                )
            ) as handoff_days

        from clean.order_items oi

        join clean.orders o 
        on oi.order_id = o.order_id

        where o.order_status = 'delivered';
        """, connection = engine
        )

In [33]:
seller_performance.head()

seller_id,order_id,delivery_days,delay_days,handoff_days
str,str,"decimal[38,0]","decimal[38,0]","decimal[38,0]"
"""dd7ddc04e1b6c2c614352b383efe2d36""","""00018f77f2f0320c557190d7a144bdd3""",16,-2,8
"""5b51032eddd242adc84c38acab88f23d""","""000229ec398224ef6ca0657da4fc703e""",7,-13,1
"""df560393f3a51e74553ab94004ba5c87""","""00042b26cf59d7ce69dfabb4e55b4fd9""",25,-15,11
"""6426d21aca402a131fc0a5d0960a3c90""","""00048cc3ae777c65dbb7d2a0634bc1ea""",6,-14,1
"""7040e82f899a04d1b434b795a43b4617""","""00054e8431b9d7675808bcb819fb4a32""",8,-16,1


In [31]:
seller_performance_summary = (
        seller_performance.group_by('seller_id')
        .agg(
            orders = pl.col('order_id').n_unique(),
            median_delivery_days = pl.col('delivery_days').median(),
            avg_delivery_days = pl.col('delivery_days').mean(),
            median_delay_days = pl.col('delay_days').median(),
            avg_delay_days = pl.col('delay_days').mean(),
            late_pct = (pl.col('delay_days') > 0).mean() * 100,
            median_handoff_days = pl.col('handoff_days').median(),
            avg_handoff_days = pl.col('handoff_days').mean()
            )
        )

In [34]:
seller_performance_summary = seller_performance_summary.filter(
        pl.col('orders') >= 30
        )

In [35]:
seller_performance_summary.head()

seller_id,orders,median_delivery_days,avg_delivery_days,median_delay_days,avg_delay_days,late_pct,median_handoff_days,avg_handoff_days
str,u32,f64,f64,f64,f64,f64,f64,f64
"""cd6efc47efaabf134f8bdb654e10b4f1""",31,6.00,8.09,-14.00,-13.12,6.25,0.00,0.97
"""8b321bb669392f5163d04c59e235e066""",930,10.00,12.17,-9.00,-9.11,8.66,1.00,1.97
"""83e197e95a1bbabc8c75e883ed016c47""",46,9.50,10.09,-13.00,-12.83,1.85,1.00,1.48
"""537eb890efff034a88679788b647c564""",158,7.00,8.67,-11.00,-11.51,3.51,1.00,1.19
"""05d2173d43ea568aa0540eba70d2ca76""",62,13.00,14.26,-11.00,-10.67,4.55,2.00,2.11


In [36]:
pl.read_database(
        query = """
        select *
        from analytics.volume_order_revenue
        limit 5;
        """, connection = engine
        )

seller_id,revenue,orders,items_sold,average_price,average_order_value
str,f64,i64,i64,f64,str
"""0015a82c2db000af6aaaf3ae2ecb0532""","2,685.00",3,3,895.00,"""895.00"""
"""001cca7ae9ae17fb1caed9dfb1094831""","25,080.03",200,239,104.94,"""125.40"""
"""001e6ad469a905060d959994f1b41e4f""",250.00,1,1,250.00,"""250.00"""
"""002100f778ceb8431b7a1020ff7ab48f""","1,234.50",51,55,22.45,"""24.21"""
"""003554e2dce176b5555353e4f3555ac8""",120.00,1,1,120.00,"""120.00"""


In [37]:
seller_revenue = pl.read_database(
        query = """
        select seller_id,
        revenue 
        from analytics.volume_order_revenue;
        """, connection = engine
        )

In [38]:
seller_revenue.head()

seller_id,revenue
str,f64
"""0015a82c2db000af6aaaf3ae2ecb0532""","2,685.00"
"""001cca7ae9ae17fb1caed9dfb1094831""","25,080.03"
"""001e6ad469a905060d959994f1b41e4f""",250.00
"""002100f778ceb8431b7a1020ff7ab48f""","1,234.50"
"""003554e2dce176b5555353e4f3555ac8""",120.00


In [39]:
seller_performance_summary = (
        seller_performance_summary.join(
            seller_revenue, on = 'seller_id', how = 'inner'
            )
        )

In [40]:
seller_performance_summary.head()

seller_id,orders,median_delivery_days,avg_delivery_days,median_delay_days,avg_delay_days,late_pct,median_handoff_days,avg_handoff_days,revenue
str,u32,f64,f64,f64,f64,f64,f64,f64,f64
"""001cca7ae9ae17fb1caed9dfb1094831""",195,11.00,12.63,-13.00,-12.27,5.13,2.00,2.14,"25,080.03"
"""002100f778ceb8431b7a1020ff7ab48f""",50,13.50,15.78,-8.00,-7.35,16.67,3.00,4.09,"1,234.50"
"""004c9cd9d87a3c30c522c48c4fc07416""",156,12.00,13.96,-12.00,-11.06,5.95,1.00,1.29,"19,712.71"
"""00ee68308b45bc5e2660cd833c3f81cc""",135,6.50,8.73,-11.00,-9.94,6.40,1.00,1.62,"20,260.00"
"""00fc707aaaad2d31347cf883cd2dfe10""",103,13.00,15.35,-16.00,-14.58,3.70,2.00,2.36,"12,684.90"


In [42]:
from scipy.stats import spearmanr

metrics = [
    "median_delivery_days",
    "late_pct",
    "median_handoff_days"
]

for metric in metrics:
    corr, p = spearmanr(
        seller_performance_summary["revenue"],
        seller_performance_summary[metric]
    )

    print(
        f"{metric}: "
        f"spearman={corr:.4f}, "
        f"p={p:.4f}"
    )

median_delivery_days: spearman=0.1741, p=0.0000
late_pct: spearman=0.0858, p=0.0317
median_handoff_days: spearman=0.0646, p=0.1061


In [43]:
seller_performance_summary = seller_performance_summary.with_columns(
    pl.col("revenue")
    .qcut(5)
    .alias("revenue_quintile")
)

In [45]:
seller_performance_summary.group_by("revenue_quintile").agg(
    pl.col("median_delivery_days").median(),
    pl.col("late_pct").median(),
    pl.col("median_handoff_days").median()
).sort("revenue_quintile")

revenue_quintile,median_delivery_days,late_pct,median_handoff_days
cat,f64,f64,f64
"""(-inf, 3920.296000000001]""",9.00,4.82,2.00
"""(10303.655999999994, 19079.472]""",9.50,5.36,2.00
"""(19079.472, inf]""",10.00,6.28,2.00
"""(3920.296000000001, 6456.680000000001]""",9.00,5.56,1.00
"""(6456.680000000001, 10303.655999999994]""",9.00,4.24,1.00


In [46]:
pl.read_database(
        query = """
        select column_name
        from information_schema.columns
        where table_schema = 'clean'
        and table_name = 'products'
        """, connection = engine
        )

column_name
str
"""product_id"""
"""product_category_name"""
"""product_name_lenght"""
"""product_description_lenght"""
"""product_photos_qty"""
"""product_weight_g"""
"""product_length_cm"""
"""product_height_cm"""
"""product_width_cm"""


In [51]:
product_level_delivery = pl.read_database(
        query = """
        select c.product_category_name_english as category,
        
        p.product_weight_g as weight,

        (p.product_length_cm * p.product_height_cm * p.product_width_cm) as volume,

        extract(
            day from (
                o.order_delivered_customer_date 
                - o.order_purchase_timestamp
                )
            ) as delivery_days,

        extract(
            day from (
                o.order_delivered_customer_date 
                - o.order_estimated_delivery_date
                )
            ) as delay_days

        from clean.order_items oi 

        join clean.orders o 
        on oi.order_id = o.order_id

        join clean.products p 
        on oi.product_id = p.product_id

        join clean.categories c 
        on p.product_category_name = c.product_category_name
        
        where o.order_status = 'delivered'
        """, connection = engine 
        )

In [52]:
product_level_delivery = product_level_delivery.drop_nulls()

In [53]:
product_level_delivery.head()

category,weight,volume,delivery_days,delay_days
str,i64,i64,"decimal[38,0]","decimal[38,0]"
"""pet_shop""","30,000","60,000",16,-2
"""furniture_decor""","3,050","14,157",7,-13
"""garden_tools""","3,750","42,000",25,-15
"""housewares""",450,"2,880",6,-14
"""telephony""",200,"2,700",8,-16


In [56]:
product_level_delivery.select(pl.col('weight'), pl.col('volume')).describe()

statistic,weight,volume
str,f64,f64
"""count""","108,629.00","108,629.00"
"""null_count""",0.00,0.00
"""mean""","2,095.64","15,221.94"
"""std""","3,744.51","23,267.66"
"""min""",0.00,168.00
"""25%""",300.00,"2,856.00"
"""50%""",700.00,"6,552.00"
"""75%""","1,800.00","18,375.00"
"""max""","40,425.00","296,208.00"


In [58]:
product_level_delivery.sort("weight", descending=True).head(10)

category,weight,volume,delivery_days,delay_days
str,i64,i64,"decimal[38,0]","decimal[38,0]"
"""bed_bath_table""","40,425","23,660",14,-4
"""bed_bath_table""","40,425","23,660",33,-3
"""bed_bath_table""","40,425","23,660",25,1
"""pet_shop""","30,000","60,000",16,-2
"""health_beauty""","30,000","140,360",12,-11
"""health_beauty""","30,000","210,000",13,-4
"""auto""","30,000","216,000",20,-5
"""housewares""","30,000","114,840",2,-13
"""sports_leisure""","30,000","155,200",20,-44


In [60]:
product_level_delivery.sort("volume", descending=True).head(10)

category,weight,volume,delivery_days,delay_days
str,i64,i64,"decimal[38,0]","decimal[38,0]"
"""housewares""","25,250","296,208",8,-15
"""furniture_living_room""","30,000","294,000",7,-7
"""furniture_living_room""","30,000","294,000",30,-7
"""furniture_living_room""","30,000","294,000",17,-21
"""furniture_living_room""","30,000","293,706",15,-2
"""furniture_living_room""","30,000","288,000",9,-9
"""furniture_living_room""","30,000","288,000",9,-10
"""furniture_living_room""","30,000","288,000",11,-6
"""furniture_living_room""","30,000","288,000",11,-6


In [63]:
for metric in ["weight", "volume"]:
    corr, p = spearmanr(
        product_level_delivery[metric],
        product_level_delivery["delivery_days"]
    )

    print(
        f"{metric} vs delivery_days: "
        f"rho={corr:.4f}, p={p:.4f}"
    )

weight vs delivery_days: rho=0.1015, p=0.0000


volume vs delivery_days: rho=0.0816, p=0.0000


In [64]:
for metric in ["weight", "volume"]:
    corr, p = spearmanr(
        product_level_delivery[metric],
        product_level_delivery["delay_days"]
    )

    print(
        f"{metric} vs delay_days: "
        f"rho={corr:.4f}, p={p:.4f}"
    )

weight vs delay_days: rho=-0.0068, p=0.0250


volume vs delay_days: rho=-0.0081, p=0.0078


In [69]:
freight_df = pl.read_database(
        query = """
        select oi.freight_value,
        extract(
            day from (
                o.order_delivered_customer_date 
                - o.order_purchase_timestamp
                )
            ) as delivery_days,

        extract(
            day from (
                o.order_delivered_customer_date 
                - o.order_estimated_delivery_date
                )
            ) as delay_days

        from clean.order_items oi 

        join clean.orders o 
        on oi.order_id = o.order_id

        where o.order_status = 'delivered';
        """, connection = engine
        )

In [70]:
freight_df.head()

freight_value,delivery_days,delay_days
f64,"decimal[38,0]","decimal[38,0]"
19.93,16,-2
17.87,7,-13
18.14,25,-15
12.69,6,-14
11.85,8,-16


In [72]:
freight_df = freight_df.drop_nulls()

In [73]:
corr, p = spearmanr(
    freight_df["freight_value"],
    freight_df["delivery_days"]
)

print(
    f"freight vs delivery_days: "
    f"rho={corr:.4f}, p={p:.4f}"
)

freight vs delivery_days: rho=0.4222, p=0.0000


In [74]:
corr, p = spearmanr(
    freight_df["freight_value"],
    freight_df["delay_days"]
)

print(
    f"freight vs delay_days: "
    f"rho={corr:.4f}, p={p:.4f}"
)

freight vs delay_days: rho=-0.1446, p=0.0000


In [75]:
freight_summary = (
    freight_df
    .with_columns(
        pl.col("freight_value")
        .qcut(5)
        .alias("freight_quintile")
    )
    .group_by("freight_quintile")
    .agg(
        pl.col("delivery_days").median().alias("median_delivery_days"),
        pl.col("delay_days").median().alias("median_delay_days")
    )
)

In [77]:
freight_summary

freight_quintile,median_delivery_days,median_delay_days
cat,f64,f64
"""(15.11, 17.86]""",11.00,-12.00
"""(12.13, 15.11]""",9.00,-12.00
"""(-inf, 12.13]""",5.00,-9.00
"""(23.28, inf]""",13.00,-13.00
"""(17.86, 23.28]""",11.00,-13.00
